In [ ]:
import pandas as pd
import torch
from transformers import pipeline
from tqdm import tqdm

In [ ]:
def classify_motivations_on_gpu(csv_file_in, csv_file_out):
    print("="*80)
    print(f"Reading csv: {csv_file_in}")
    print("="*80)

    try:
        df = pd.read_csv(csv_file_in)
        print("✅ CSV loaded successfully.")
    except Exception as e:
        print(f"❌ Error loading CSV: {e}")
        return

    # Extract mentor motivation column based on mentor identity
    df['mentor_motivation'] = df.apply(
        lambda row: row['person1_motivation_mentorship']
        if str(row['person1_is_mentor']).lower() == 'true'
        else row['person2_motivation_mentorship'],
        axis=1
    )

    # Clean and prepare text for model
    df["mentor_motivation_str"] = df["mentor_motivation"].astype(str).fillna("")
    non_empty_rows = df[df["mentor_motivation_str"] != ""]
    text_to_classify = non_empty_rows["mentor_motivation_str"].tolist()

    print(f"Found {len(text_to_classify)} items to classify." if len(text_to_classify) else "⚠️ No text to classify.")

    # Detect GPU or CPU
    device_to_use = 0 if torch.cuda.is_available() else -1
    print(f"✅ Using GPU: {torch.cuda.get_device_name(0)}" if device_to_use == 0 else "⚠️ Using CPU (slower)")

    # Load XLM-R model
    try:
        classifier = pipeline(
            "zero-shot-classification",
            model="joeddav/xlm-roberta-large-xnli",
            device=device_to_use,
            batch_size=4
        )
    except Exception as e:
        print(f"❌ Error loading model: {e}")
        return

    candidate_labels = ["intrinsic motivation", "extrinsic motivation"]
    hypothesis_template = "This text is about {}."

    results = []
    for text in tqdm(text_to_classify, desc="Classifying with XLM-R (handling errors)"):
        try:
            result = classifier(text, candidate_labels, hypothesis_template=hypothesis_template)
            results.append({"label": result["labels"][0], "score": result["scores"][0]})
        except:
            results.append({"label": "None", "score": 0.0})

    df.loc[non_empty_rows.index, 'motivation'] = [r['label'] for r in results]
    df.loc[non_empty_rows.index, 'confidence'] = [r['score'] for r in results]

    # Fill empty values
    df['motivation'] = df['motivation'].fillna("None")
    df['confidence'] = df['confidence'].fillna(0.0)

    # Save output file
    df.to_csv(csv_file_out, index=False)
    print(f"✅ Results saved to {csv_file_out}")


In [ ]:
INPUT_FILE = "merged_conversations_with_translations.csv"  # Change if needed
OUTPUT_FILE = "final_result_xlm_roberta.csv"

classify_motivations_on_gpu(INPUT_FILE, OUTPUT_FILE,)


Reading csv: merged_conversations_with_translations.csv
❌ Error loading CSV: 'utf-8' codec can't decode byte 0xd9 in position 0: unexpected end of data


In [ ]:
import re
import torch
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoModelForSequenceClassification

tokenizer = AutoTokenizer.from_pretrained("joeddav/xlm-roberta-large-xnli")
model = AutoModelForSequenceClassification.from_pretrained("joeddav/xlm-roberta-large-xnli")
model.eval()

# Intrinsic keyword regex (expandable)
intrinsic_keywords = [
    r"\blearn", r"\blearning", r"\bcurios", r"\bimprov", r"\bskill", r"\bexperience", r"\bgrow", r"\bgrowth",
    r"\bpersonal development", r"\bdevelop my", r"\bdevelop myself", r"\bknowledge", r"\bstudy",
    r"\bperspective", r"\bangles of a", r"\bopen my perspective", r"\bopen my"
]
intrinsic_pattern = re.compile("|".join(intrinsic_keywords), flags=re.IGNORECASE)

label_map = {0: "INTRINSIC", 1: "EXTRINSIC"}     # or whatever your model mapping is
# If 3-class model, include UNKNOWN mapping

def predict_with_postprocess(text):
    # model logits -> probabilities
    inputs = tokenizer(text, truncation=True, padding=True, return_tensors="pt")
    with torch.no_grad():
        out = model(**inputs)
        logits = out.logits.squeeze(0)
        probs = F.softmax(logits, dim=-1).cpu().numpy()

    # map probs to labels (example for 2-class model)
    p_intr = float(probs[0])
    p_extr = float(probs[1])
    pred = "INTRINSIC" if p_intr > p_extr else "EXTRINSIC"
    confidence = max(p_intr, p_extr)

    # rule-based fallback: if intrinsic keywords are present and model either predicted EXTRINSIC with low confidence
    # OR model gave non-trivial intrinsic probability, flip to INTRINSIC
    if intrinsic_pattern.search(text):
        # conservative flip rule:
        if pred == "EXTRINSIC" and confidence < 0.80:
            pred = "INTRINSIC"
            confidence = p_intr
        # or soft flip if model at least assigns some mass to intrinsic (tunable threshold)
        elif p_intr >= 0.30 and p_intr > p_extr:
            pred = "INTRINSIC"
            confidence = p_intr

    return {"label": pred, "p_intrinsic": p_intr, "p_extrinsic": p_extr, "confidence": confidence}

Some weights of the model checkpoint at joeddav/xlm-roberta-large-xnli were not used when initializing XLMRobertaForSequenceClassification: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
- This IS expected if you are initializing XLMRobertaForSequenceClassification from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing XLMRobertaForSequenceClassification from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).


In [ ]:
import pandas as pd

# Load classified results CSV
df = pd.read_csv("final_result_xlm_roberta.csv")

# Select only useful columns (ID + motivation)
filtered_df = df[[
    'person1_id',
    'person2_id',
    'conversation_start',
    'mentor_motivation_str',
    'motivation',
    'confidence'
]]

# Save to a clean output file
filtered_df.to_csv("motivation_clean_output.csv", index=False)

print("✅ Clean file saved: motivation_clean_output.csv")
filtered_df.head()


✅ Clean file saved: motivation_clean_output.csv


,person1_id,person2_id,conversation_start,mentor_motivation_str,motivation,confidence
0,1635114,1880293,2025-06-09 23:02:06.999 +0530,to learn from,extrinsic motivation,0.629855
1,310682,1899553,2025-06-20 19:11:54.805 +0530,NaN,extrinsic motivation,0.588303
2,310682,1860596,2025-04-15 06:51:03.830 +0530,NaN,extrinsic motivation,0.588303
3,310682,1360170,2025-06-02 20:30:39.580 +0530,NaN,extrinsic motivation,0.588303
4,310682,1889323,2025-06-07 04:03:24.605 +0530,NaN,extrinsic motivation,0.588303


In [ ]:
low_conf_df = filtered_df[filtered_df['confidence'] < 0.5]
low_conf_df.to_csv("motivation_low_confidence_clean.csv", index=False)
print("✅ Saved low confidence output only with ID + motivation columns")
print(f"Rows with confidence < 50%: {len(low_conf_df)}")

✅ Saved low confidence output only with ID + motivation columns
Rows with confidence < 50%: 0


In [ ]:
%pip install langdetect

In [ ]:
# ====================================================
#  HIGH-ACCURACY INTRINSIC vs EXTRINSIC CLASSIFICATION
# ====================================================

from transformers import pipeline
import pandas as pd, re, time
from tqdm import tqdm
from langdetect import detect

# 1️⃣ Load multilingual model (robust + accurate)
classifier = pipeline(
    "zero-shot-classification",
    model="joeddav/xlm-roberta-large-xnli",  # multilingual, accurate
    device=0  # use GPU if available
)
# 2️⃣ Define the prompt template
FINAL_PROMPT_TEMPLATE = """
You are a psychologist specializing in human motivation.
Classify the motivation in the following text as INTRINSIC, EXTRINSIC, or UNKNOWN.

Definitions:
- INTRINSIC: Focuses on curiosity, learning, growth, self-improvement, or personal satisfaction.
- EXTRINSIC: Focuses on rewards, recognition, helping others, impact, or societal outcomes.

Output only one word: INTRINSIC, EXTRINSIC, or UNKNOWN.

Text: "{}"
Answer:
"""


# 3️⃣ Candidate labels
labels = ["Intrinsic motivation", "Extrinsic motivation"]

# 4️⃣ Load your existing model output or dataset
df = pd.read_csv("motivation_clean_output.csv")

# 5️⃣ Text cleaning
def clean_text(txt):
    if pd.isna(txt): return ""
    txt = re.sub(r'\s+', ' ', str(txt)).strip()
    return txt

# 6️⃣ Create prompt
def make_prompt(motivation_text):
    try:
        _ = detect(motivation_text)
    except:
        pass
    return FINAL_PROMPT_TEMPLATE.format(motivation_text)

# 7️⃣ Run classification (accurate but slower)
tqdm.pandas()
results = []
for text in tqdm(df["mentor_motivation_str"].fillna(""), desc="Refining motivation labels"):
    t = clean_text(text)
    if not t:
        results.append({"label": "Extrinsic motivation", "score": 0.0})
        continue
    prompt = make_prompt(t)
    res = classifier(prompt, candidate_labels=labels)
    label = res["labels"][0]
    score = res["scores"][0]
    results.append({"label": label, "score": score})
    # time.sleep(0.1)  # slow but stable for accurate evaluation

# 8️⃣ Add predictions
df["refined_motivation"] = [r["label"] for r in results]
df["refined_confidence"] = [r["score"] for r in results]

# 9️⃣ Apply smart correction for confidently wrong cases
def correction_rule(text, label, conf):
    # If confidence is below 0.65, classify as UNKNOWN
    if conf < 0.65:
        return "UNKNOWN"

    text = str(text).lower()
    intrinsic_kw = [
        "learn","learning","improve","growth","mentor","values","curious",
        "explore","perspective","purpose","help","impact","develop","creative"
    ]
    extrinsic_kw = [
        "salary","money","bonus","promotion","recognition","certificate",
        "award","profit","funding","customers","validation","title"
    ]
    i = sum(1 for k in intrinsic_kw if k in text)
    e = sum(1 for k in extrinsic_kw if k in text)

    # If the initial label is Extrinsic and text contains intrinsic keywords
    intrinsic_keywords_user = ["learn", "growth", "improve", "develop", "curiosity", "perspective"]
    if any(k in text for k in intrinsic_keywords_user) and label.lower().startswith("extrinsic"):
        return "Intrinsic motivation"

    # if strong intrinsic signals but labeled extrinsic → correct it
    if label.lower().startswith("extrinsic") and i > e and i >= 2:
        return "Intrinsic motivation"
    # if very low confidence → fallback to intrinsic if intrinsic signals dominate
    if conf < 0.6 and i > e:
        return "Intrinsic motivation"

    return label

df["final_motivation"] = df.apply(lambda r: correction_rule(
    r["mentor_motivation_str"], r["refined_motivation"], r["refined_confidence"]
), axis=1)

# 10️⃣ Save final refined output
df.to_csv("Final.csv", index=False)

# 11️⃣ Summary
print("✅ Saved refined file: Final.csv")
print(df["final_motivation"].value_counts())

Some weights of the model checkpoint at joeddav/xlm-roberta-large-xnli were not used when initializing XLMRobertaForSequenceClassification: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
- This IS expected if you are initializing XLMRobertaForSequenceClassification from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing XLMRobertaForSequenceClassification from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).
Device set to use cuda:0
Refining motivation labels: 100%|██████████| 4999/4999 [00:40<00:00, 124.64it/s]

✅ Saved refined file: Final.csv
final_motivation
UNKNOWN    4999
Name: count, dtype: int64
